# Tech Challenge — Fase 1: Olist E-commerce (Crescimento e Receita)

Relatório executivo para investidores, com base no *Brazilian E-Commerce Public Dataset by Olist* (≈100 mil pedidos, 2016–2018).

## Governança: fonte, regras de negócio e reprodutibilidade

**Fonte:** Brazilian E-Commerce Public Dataset by Olist (Kaggle). Arquivos usados: customers, orders, order_items, payments, reviews, products, sellers, geolocation e category_translation.

**Regras de negócio adotadas (todas aplicadas no código):**
- Receita = soma de `price` dos itens, **sem frete**.
- Considera apenas pedidos com `order_status == "delivered"` e com pelo menos um item.
- Comparações mensais usam **jan/2017 a ago/2018**: 2016 tem pouquíssimos pedidos e set–out/2018 estão incompletos.
- Ticket médio = receita ÷ pedidos entregues com itens.

**Qualidade e integridade:** chaves primárias sem duplicidade (seção 2). Junções com `validate=` e `assert` para garantir que nenhum item ou pedido foi duplicado (seções 3–5). Receita dos itens conferida contra a receita agrupada por pedido.

**Reprodutibilidade:** coloque os CSVs na mesma pasta do notebook e execute de cima para baixo (*Executar tudo*). Bibliotecas: pandas, numpy, scipy, matplotlib. Não há componentes aleatórios.

**Limitações:** dataset histórico encerrado em 2018 (sem dados externos como IPCA/IBGE incorporados). A previsão se baseia em poucos meses e só um Black Friday foi observado.

## 1. Importação das bases e principais bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

customers = pd.read_csv("olist_customers_dataset.csv")
orders = pd.read_csv("olist_orders_dataset.csv")
order_items = pd.read_csv("olist_order_items_dataset.csv")
payments = pd.read_csv("olist_order_payments_dataset.csv")
reviews = pd.read_csv("olist_order_reviews_dataset.csv")
products = pd.read_csv("olist_products_dataset.csv")
sellers = pd.read_csv("olist_sellers_dataset.csv")
geolocation = pd.read_csv("olist_geolocation_dataset.csv")
category_translation = pd.read_csv("product_category_name_translation.csv")

## 2. Conferir as chaves e preparar as datas

In [ ]:
for nome, tabela, chave in [
    ("orders", orders, "order_id"),
    ("customers", customers, "customer_id"),
    ("products", products, "product_id"),
    ("sellers", sellers, "seller_id"),
    ("category_translation", category_translation, "product_category_name"),
]:
    qtd_duplicadas = tabela[chave].duplicated().sum()
    print(f"{nome}: {qtd_duplicadas} chaves duplicadas em {chave}")
    assert qtd_duplicadas == 0, f"Revisar duplicações em {nome}.{chave}"

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"], errors="coerce"
)
print("Primeira compra:", orders["order_purchase_timestamp"].min())
print("Última compra:", orders["order_purchase_timestamp"].max())

## 3. Criar a base de itens: produtos, categorias, estados e sellers

In [ ]:
itens = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_status", "order_purchase_timestamp"]],
        on="order_id", how="left", validate="many_to_one"
    )
    .merge(
        customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
        on="customer_id", how="left", validate="many_to_one"
    )
    .merge(
        products[["product_id", "product_category_name"]],
        on="product_id", how="left", validate="many_to_one"
    )
    .merge(
        sellers[["seller_id", "seller_city", "seller_state"]],
        on="seller_id", how="left", validate="many_to_one"
    )
    .merge(category_translation, on="product_category_name",
           how="left", validate="many_to_one")
)
itens["categoria"] = (
    itens["product_category_name_english"]
    .fillna(itens["product_category_name"])
    .fillna("sem categoria")
)
itens["receita_produtos"] = itens["price"]
itens["ano_mes"] = itens["order_purchase_timestamp"].dt.to_period("M")

assert len(itens) == len(order_items), "O relacionamento alterou a quantidade de itens."
print("Itens na origem:", len(order_items), "| Itens após as conexões:", len(itens))
display(itens[["order_id", "product_id", "seller_id", "customer_state",
               "customer_city", "categoria", "price"]].head())

## 4. Criar a base de pedidos: faturamento e ticket médio

Para calcular ticket médio, precisamos de **um valor total para cada pedido**. Como order_items tem várias linhas por pedido, fazemos groupby("order_id") antes de juntar com orders.

 sum(price) = receita de produtos do pedido (sem frete).
 sum(freight_value) = frete associado aos itens.
 size() = quantidade de linhas de itens.

Depois, conectamos clientes para obter UF e município. A base final terá exatamente **uma linha por pedido**.

In [ ]:
itens_agrupados = (
    order_items.groupby("order_id", as_index=False)
    .agg(
        receita_produtos=("price", "sum"),
        frete_total=("freight_value", "sum"),
        qtd_itens=("order_item_id", "size"),
    )
)

pedidos = (
    orders.merge(
        customers[["customer_id", "customer_unique_id", "customer_city", "customer_state"]],
        on="customer_id", how="left", validate="many_to_one"
    )
    .merge(itens_agrupados, on="order_id", how="left", validate="one_to_one")
)
pedidos["ano_mes"] = pedidos["order_purchase_timestamp"].dt.to_period("M")

assert len(pedidos) == len(orders), "O relacionamento alterou a quantidade de pedidos."
assert pedidos["order_id"].is_unique, "order_id repetido na base de pedidos!"
print("Pedidos na origem:", len(orders), "| Pedidos após os JOINs:", len(pedidos))
display(pedidos[["order_id", "customer_state", "order_status",
                 "receita_produtos", "frete_total"]].head())

## 5. Validar os relacionamentos

In [ ]:
validacao = pd.DataFrame([
    ["Itens originais", len(order_items)],
    ["Itens consolidados", len(itens)],
    ["Pedidos originais", len(orders)],
    ["Pedidos consolidados", len(pedidos)],
    ["Pedidos sem itens associados", int(pedidos["receita_produtos"].isna().sum())],
    ["Pedidos com data de compra ausente", int(pedidos["order_purchase_timestamp"].isna().sum())],
    ["Itens sem estado do cliente", int(itens["customer_state"].isna().sum())],
], columns=["Verificação", "Quantidade"])
display(validacao)

# Comparar receita total de itens com receita total agrupada em pedidos
assert abs(order_items["price"].sum() - itens_agrupados["receita_produtos"].sum()) < 0.01
print("Validação financeira concluída: sem duplicação dos preços dos itens.")

## 6. Definir a regra de negócio e os indicadores principais

 Filtramos apenas pedidos com status delivered. A receita é a soma de preço desses pedidos, sem frete; o ticket médio é a receita dividida pelo número de pedidos entregues com itens.

In [ ]:
itens_entregues = itens.loc[itens["order_status"].eq("delivered")].copy()
pedidos_entregues = pedidos.loc[
    pedidos["order_status"].eq("delivered") & pedidos["receita_produtos"].notna()
].copy()

receita_total = pedidos_entregues["receita_produtos"].sum()
qtd_pedidos = pedidos_entregues["order_id"].nunique()
ticket_medio = receita_total / qtd_pedidos
qtd_clientes = pedidos_entregues["customer_unique_id"].nunique()

assert abs(receita_total - itens_entregues["price"].sum()) < 0.01
print(f"Receita de produtos sem frete: R$ {receita_total:,.2f}")
print(f"Pedidos entregues com itens: {qtd_pedidos:,}")
print(f"Ticket médio: R$ {ticket_medio:,.2f}")
print(f"Clientes únicos: {qtd_clientes:,}")

## 7. Analise 1 - Evolução mensal: pedidos, receita e ticket médio

Agrupamos a base de pedidos por mês de compra, para que cada pedido conte uma única vez. Por padronização, a comparação mensal usa janeiro/2017 a agosto/2018, evitando os meses extremos incompletos do dataset.

- Pedidos: quantidade de pedidos entregues.
- Receita: soma do preço dos itens desses pedidos, sem frete.
- Ticket médio: receita mensal ÷ número de pedidos mensais.
- Crescimento mensal: variação percentual da receita contra o mês anterior.

 Atribuir pedidos entregues ao mês da compra permite comparar safras de compra, mas não representa o momento contábil do reconhecimento da receita.

In [ ]:
mensal = (
    pedidos_entregues.groupby("ano_mes", as_index=False)
    .agg(
        pedidos=("order_id", "nunique"),
        receita=("receita_produtos", "sum")
    )
    .sort_values("ano_mes")
)
mensal["ticket_medio"] = mensal["receita"] / mensal["pedidos"]
mensal_comparavel = mensal.loc[
    mensal["ano_mes"].between(pd.Period("2017-01"), pd.Period("2018-08"))
].copy()
mensal_comparavel["crescimento_receita_pct"] = (
    mensal_comparavel["receita"].pct_change() * 100
)
mensal_comparavel["mes"] = mensal_comparavel["ano_mes"].astype(str)
display(mensal_comparavel.tail(8))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(mensal_comparavel["mes"], mensal_comparavel["pedidos"], color="#2563EB")
ax.set(title="Pedidos por mês de compra — pedidos entregues", xlabel="Mês", ylabel="Pedidos")
ax.tick_params(axis="x", rotation=75)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mensal_comparavel["mes"], mensal_comparavel["receita"], marker="o", color="#0F766E")
ax.set(title="Receita mensal de produtos (sem frete)", xlabel="Mês", ylabel="Receita (R$)")
ax.tick_params(axis="x", rotation=75)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(mensal_comparavel["mes"], mensal_comparavel["ticket_medio"], marker="o", color="#7C3AED")
ax.set(title="Ticket médio mensal", xlabel="Mês", ylabel="R$ por pedido")
ax.tick_params(axis="x", rotation=75)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 8. Estatística descritiva por pedido

### 8.1 Estatística descritiva por pedido

Cada linha de `pedidos_entregues` é um pedido. Calculamos, para receita de produtos, frete, valor total (produtos + frete) e quantidade de itens:

- **Tendência central:** média, mediana, moda e média aparada (5%).
- **Posição:** mínimo, quartis, P90, P99 e máximo.
- **Variabilidade/dispersão:** amplitude, intervalo interquartil (IQR), variância, desvio padrão e coeficiente de variação.
- **Forma da distribuição:** assimetria e curtose.
- **Outliers:** pedidos fora de Q1 − 1,5·IQR e Q3 + 1,5·IQR.

Variância e desvio padrão são amostrais (ddof=1, padrão do pandas).

In [ ]:
pedidos_entregues["valor_total"] = (
    pedidos_entregues["receita_produtos"] + pedidos_entregues["frete_total"]
)

variaveis = {
    "receita_produtos": "Receita de produtos (R$)",
    "frete_total": "Frete (R$)",
    "valor_total": "Valor total do pedido (R$)",
    "qtd_itens": "Itens por pedido",
}

def resumo_estatistico(s):
    s = s.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    limite_inf, limite_sup = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return pd.Series({
        # tendência central
        "n pedidos": s.count(),
        "média": s.mean(),
        "mediana": s.median(),
        "moda": s.mode().iloc[0],
        "média aparada 5%": stats.trim_mean(s, 0.05),
        # posição
        "mínimo": s.min(),
        "Q1 (25%)": q1,
        "Q3 (75%)": q3,
        "P90": s.quantile(0.90),
        "P99": s.quantile(0.99),
        "máximo": s.max(),
        # variabilidade / dispersão
        "amplitude": s.max() - s.min(),
        "IQR": iqr,
        "variância": s.var(),
        "desvio padrão": s.std(),
        "coef. de variação (%)": s.std() / s.mean() * 100,
        # forma
        "assimetria": s.skew(),
        "curtose": s.kurt(),
        # outliers
        "outliers (regra do IQR)": int(((s < limite_inf) | (s > limite_sup)).sum()),
        "outliers (%)": ((s < limite_inf) | (s > limite_sup)).mean() * 100,
    })

estatisticas_pedidos = pd.DataFrame({
    nome: resumo_estatistico(pedidos_entregues[col])
    for col, nome in variaveis.items()
})

display(estatisticas_pedidos.round(2))

### Como ler a tabela

- **Média > mediana** e **assimetria positiva** indicam cauda à direita: poucos pedidos caros puxam a média para cima. Nesse caso a **mediana** representa melhor o pedido típico.
- **Coeficiente de variação** acima de ~50% indica alta dispersão relativa.
- **Curtose alta** indica caudas pesadas (muitos valores extremos).
- O ticket médio de R\$ 137,04 usado nas análises mensais é a média, e a mediana mostra o valor do pedido mais comum.
- **Itens por pedido:** como ~90% dos pedidos têm 1 item, o IQR é 0 e a regra do IQR marca todo pedido com 2+ itens como "outlier". Para essa variável, use a tabela de distribuição (7.3) em vez da contagem de outliers.

### 8.2 Distribuição visual: histograma e boxplot

Como as distribuições são assimétricas, o histograma é cortado no P99 para não ser dominado por valores extremos (os outliers continuam contados na tabela).

In [ ]:
cores = {"receita_produtos": "#2563EB", "frete_total": "#0F766E",
         "valor_total": "#7C3AED"}

fig, eixos = plt.subplots(2, 3, figsize=(15, 8))

for j, (col, cor) in enumerate(cores.items()):
    s = pedidos_entregues[col].dropna()
    p99 = s.quantile(0.99)
    media, mediana = s.mean(), s.median()

    ax = eixos[0, j]
    ax.hist(s[s <= p99], bins=50, color=cor, alpha=0.85)
    ax.axvline(media, color="red", linestyle="--", label=f"Média: {media:,.1f}")
    ax.axvline(mediana, color="black", linestyle="-", label=f"Mediana: {mediana:,.1f}")
    ax.set(title=f"{variaveis[col]} (até P99)", xlabel="R$", ylabel="Pedidos")
    ax.legend()
    ax.grid(alpha=0.2)

    ax = eixos[1, j]
    ax.boxplot(s, vert=False, showfliers=True,
               flierprops=dict(marker=".", markersize=2, alpha=0.3))
    ax.set(title=f"Boxplot - {variaveis[col]}", xlabel="R$")
    ax.set_xscale("log")
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()

> Os boxplots usam escala logarítmica no eixo X para que os outliers não achatem a caixa.

### 8.3 Quantidade de itens por pedido

In [ ]:
dist_itens = (
    pedidos_entregues["qtd_itens"].astype(int).value_counts().sort_index()
    .rename("pedidos").to_frame()
)
dist_itens["participacao_pct"] = dist_itens["pedidos"] / dist_itens["pedidos"].sum() * 100
dist_itens["acumulado_pct"] = dist_itens["participacao_pct"].cumsum()
display(dist_itens.head(8).round(2))

fig, ax = plt.subplots(figsize=(10, 4))
topo = dist_itens.head(8)
ax.bar(topo.index.astype(str), topo["pedidos"], color="#2563EB")
ax.set(title="Pedidos por quantidade de itens (até 8 itens)",
       xlabel="Itens no pedido", ylabel="Pedidos")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

### 8.4 Frete como proporção do pedido

Mostra quanto o frete pesa no valor final, indicador útil para avaliar sensibilidade do cliente a preço de entrega.

In [ ]:
pedidos_entregues["frete_pct_pedido"] = (
    pedidos_entregues["frete_total"] / pedidos_entregues["valor_total"] * 100
)
display(resumo_estatistico(pedidos_entregues["frete_pct_pedido"]).round(2).to_frame("Frete / valor total (%)"))

## 9. Análise 2 — Participação por categoria e região

In [ ]:
categorias = (
    itens_entregues.groupby("categoria", as_index=False)
    .agg(receita=("price", "sum"), pedidos=("order_id", "nunique"),
         unidades=("order_item_id", "size"))
    .sort_values("receita", ascending=False)
)
categorias["participacao_pct"] = categorias["receita"] / receita_total * 100

estados = (
    pedidos_entregues.groupby("customer_state", as_index=False)
    .agg(receita=("receita_produtos", "sum"), pedidos=("order_id", "nunique"))
    .sort_values("receita", ascending=False)
)
estados["participacao_pct"] = estados["receita"] / receita_total * 100

municipios = (
    pedidos_entregues.groupby(["customer_state", "customer_city"], as_index=False)
    .agg(receita=("receita_produtos", "sum"), pedidos=("order_id", "nunique"))
    .sort_values("receita", ascending=False)
)
municipios["participacao_pct"] = municipios["receita"] / receita_total * 100

print("Categorias com maior receita:")
display(categorias.head(10))
print("Estados dos clientes com maior receita:")
display(estados.head(10))
print("Municípios dos clientes com maior receita:")
display(municipios.head(10))

In [ ]:
# Concentração de receita por categoria (Pareto)
cat_p = categorias.copy()
cat_p["acumulado_pct"] = cat_p["participacao_pct"].cumsum()
n80 = int((cat_p["acumulado_pct"] < 80).sum() + 1)
print(f"{n80} de {len(cat_p)} categorias ({n80/len(cat_p):.0%}) concentram 80% da receita.")

top = cat_p.head(15)
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(top["categoria"], top["receita"] / 1e6, color="#2563EB")
ax.set(title="Pareto de categorias: receita e % acumulado (top 15)", ylabel="Receita (R$ milhões)")
ax.tick_params(axis="x", rotation=60)
plt.setp(ax.get_xticklabels(), ha="right")
ax2 = ax.twinx()
ax2.plot(top["categoria"], top["acumulado_pct"], color="#DC2626", marker="o")
ax2.set_ylim(0, 100)
ax2.set_ylabel("% acumulado da receita")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Estados: participação na receita e ticket médio (estados com ao menos 300 pedidos)
estados["ticket_medio"] = estados["receita"] / estados["pedidos"]
est_graf = estados[estados["pedidos"] >= 300].copy()

fig, eixos = plt.subplots(1, 2, figsize=(14, 5))
t10 = estados.head(10).iloc[::-1]
eixos[0].barh(t10["customer_state"], t10["participacao_pct"], color="#2563EB")
eixos[0].set(title="Participação na receita por estado do cliente (top 10)", xlabel="% da receita")
eixos[0].grid(axis="x", alpha=0.2)

tk = est_graf.sort_values("ticket_medio").iloc[::-1]
tk = pd.concat([tk.head(8), tk.tail(8)]).drop_duplicates("customer_state").iloc[::-1]
eixos[1].barh(tk["customer_state"], tk["ticket_medio"], color="#0F766E")
eixos[1].axvline(ticket_medio, color="red", linestyle="--", label=f"Média geral R$ {ticket_medio:,.0f}")
eixos[1].set(title="Ticket médio por estado (8 maiores e 8 menores)", xlabel="R$ por pedido")
eixos[1].legend()
eixos[1].grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Os 3 maiores estados (SP, RJ, MG) concentram {estados.head(3)['participacao_pct'].sum():.1f}% da receita.")

## 10. Análise 3 — Produtos e sellers com melhor desempenho

Usamos a base de itens para somar price por product_id e seller_id. order_item_id identifica a posição do item dentro do pedido, portanto size() conta linhas de itens (unidades vendidas no dataset).

Ao agrupar sellers por seller_state, analisamos a localização do vendedor, não a localização do comprador. Rankings de receita devem ser complementados com informações de margem, avaliação e logística antes de decisões de investimento.

In [ ]:
top_produtos = (
    itens_entregues.groupby("product_id", as_index=False)
    .agg(receita=("price", "sum"), unidades=("order_item_id", "size"),
         pedidos=("order_id", "nunique"))
    .sort_values("receita", ascending=False)
)

top_sellers = (
    itens_entregues.groupby(["seller_id", "seller_state"], as_index=False)
    .agg(receita=("price", "sum"), unidades=("order_item_id", "size"),
         pedidos=("order_id", "nunique"))
    .sort_values("receita", ascending=False)
)

display(top_produtos.head(10))
display(top_sellers.head(10))

In [ ]:
import matplotlib.pyplot as plt

# Selecionar os 10 vendedores com maior receita
top10 = top_sellers.head(10).copy()

# Conectar com a base sellers
top10 = top10.merge(
    sellers[["seller_id", "seller_city", "seller_state"]],
    on=["seller_id", "seller_state"],
    how="left",
    validate="one_to_one"
)

# Criar identificação amigável
top10["nome_vendedor"] = [
    f"Vendedor {i}" for i in range(1, len(top10) + 1)
]

# Acrescentar cidade e estado
top10["identificacao"] = (
    top10["nome_vendedor"]
    + " - "
    + top10["seller_city"].fillna("Não informado").str.title()
    + " ("
    + top10["seller_state"].fillna("?")
    + ")"
)

# Criar gráfico
top10 = top10.iloc[::-1]

fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(
    top10["identificacao"],
    top10["receita"],
    color="#7C3AED"
)

ax.set_title("Top 10 vendedores por faturamento")
ax.set_xlabel("Receita de produtos (R$)")
ax.grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# Concentração de sellers
ts = top_sellers.copy()
ts["participacao_pct"] = ts["receita"] / receita_total * 100
ts["acumulado_pct"] = ts["participacao_pct"].cumsum()
n_sellers = len(ts)

def parcela_top(pct):
    return ts["participacao_pct"].head(int(round(n_sellers * pct))).sum()

resumo_sellers = pd.DataFrame({
    "Indicador": ["Sellers com vendas entregues", "Receita dos 10 maiores sellers (%)",
                  "Receita dos 1% maiores sellers (%)", "Receita dos 10% maiores sellers (%)",
                  "Sellers necessários para 80% da receita"],
    "Valor": [n_sellers, ts["participacao_pct"].head(10).sum(), parcela_top(0.01), parcela_top(0.10),
              int((ts["acumulado_pct"] < 80).sum() + 1)]
})
display(resumo_sellers.round(2))

In [ ]:
# Onde estão os sellers x onde estão os clientes (receita de itens entregues)
sel_uf = itens_entregues.groupby("seller_state")["price"].sum() / receita_total * 100
cli_uf = itens_entregues.groupby("customer_state")["price"].sum() / receita_total * 100
comparacao_uf = pd.DataFrame({"% receita por UF do seller": sel_uf, "% receita por UF do cliente": cli_uf}).fillna(0)
display(comparacao_uf.sort_values("% receita por UF do seller", ascending=False).head(6).round(1))

mesmo_estado = itens_entregues["seller_state"].eq(itens_entregues["customer_state"])
print(f"Receita em que seller e cliente estão no mesmo estado: {itens_entregues.loc[mesmo_estado, 'price'].sum() / receita_total:.1%}")

frete_comp = (
    itens_entregues.assign(mesmo_estado=mesmo_estado)
    .groupby("mesmo_estado")
    .agg(frete_medio=("freight_value", "mean"), preco_medio=("price", "mean"))
)
frete_comp["frete_sobre_preco_pct"] = frete_comp["frete_medio"] / frete_comp["preco_medio"] * 100
frete_comp.index = frete_comp.index.map({True: "Mesmo estado", False: "Estados diferentes"})
display(frete_comp.round(2))

In [ ]:
# Clientes: recompra
pedidos_por_cliente = pedidos_entregues.groupby("customer_unique_id")["order_id"].nunique()
taxa_recompra = (pedidos_por_cliente > 1).mean() * 100
clientes_recorrentes = pedidos_por_cliente[pedidos_por_cliente > 1].index
receita_recorrentes = pedidos_entregues.loc[
    pedidos_entregues["customer_unique_id"].isin(clientes_recorrentes), "receita_produtos"
].sum()
print(f"Clientes que compraram mais de uma vez: {taxa_recompra:.1f}%")
print(f"Receita gerada por esses clientes: {receita_recorrentes / receita_total:.1%} do total")

## 11. Análise 4 — Crescimento ano contra ano e sazonalidade

Comparamos os mesmos meses de 2017 e 2018 (jan–ago, únicos meses completos nos dois anos) para separar crescimento real de sazonalidade, e olhamos o efeito da Black Friday e do dia da semana.

In [ ]:
# Crescimento ano contra ano (jan–ago)
m_idx = mensal.set_index("ano_mes")
a17 = m_idx.loc["2017-01":"2017-08"]
a18 = m_idx.loc["2018-01":"2018-08"]

yoy = pd.DataFrame({
    "Pedidos": [a17["pedidos"].sum(), a18["pedidos"].sum()],
    "Receita (R$)": [a17["receita"].sum(), a18["receita"].sum()],
}, index=["Jan–Ago 2017", "Jan–Ago 2018"])
yoy["Ticket médio (R$)"] = yoy["Receita (R$)"] / yoy["Pedidos"]
display(yoy.round(2))
print(f"Crescimento da receita: {a18['receita'].sum() / a17['receita'].sum() - 1:.1%}")
print(f"Crescimento dos pedidos: {a18['pedidos'].sum() / a17['pedidos'].sum() - 1:.1%}")
print(f"Variação do ticket médio: {yoy['Ticket médio (R$)'].iloc[1] / yoy['Ticket médio (R$)'].iloc[0] - 1:.1%}")

# Receita mensal sobreposta: 2017 x 2018
fig, ax = plt.subplots(figsize=(11, 4.5))
meses = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
r17 = m_idx.loc["2017-01":"2017-12", "receita"].values / 1e3
r18 = m_idx.loc["2018-01":"2018-08", "receita"].values / 1e3
ax.plot(meses, r17, marker="o", color="#94A3B8", label="2017")
ax.plot(meses[:8], r18, marker="o", color="#0F766E", label="2018")
ax.set(title="Receita mensal: 2017 x 2018", ylabel="Receita (R$ mil)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

In [ ]:
# Black Friday 2017 (24/11/2017): pedidos por dia em novembro
ped_dia = (
    pedidos_entregues.assign(dia=pedidos_entregues["order_purchase_timestamp"].dt.normalize())
    .groupby("dia").agg(pedidos=("order_id", "nunique"), receita=("receita_produtos", "sum"))
)
nov = ped_dia.loc["2017-11-01":"2017-11-30"]
bf = pd.Timestamp("2017-11-24")
base_diaria = ped_dia.loc["2017-11-10":"2017-11-23", "pedidos"].mean()

fig, ax = plt.subplots(figsize=(12, 4))
cores_dia = ["#DC2626" if d == bf else "#2563EB" for d in nov.index]
ax.bar(nov.index.strftime("%d"), nov["pedidos"], color=cores_dia)
ax.axhline(base_diaria, color="black", linestyle="--", label=f"Média diária 10–23/nov: {base_diaria:,.0f}")
ax.set(title="Pedidos por dia em novembro de 2017 (Black Friday em vermelho)", xlabel="Dia", ylabel="Pedidos")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Black Friday: {nov.loc[bf, 'pedidos']:,.0f} pedidos = {nov.loc[bf, 'pedidos'] / base_diaria:.1f}x a média diária anterior")
print(f"Receita do dia: R$ {nov.loc[bf, 'receita']:,.0f}")
print(f"Receita de nov/2017 vs out/2017: {m_idx.loc['2017-11', 'receita'] / m_idx.loc['2017-10', 'receita'] - 1:+.1%}")
print(f"Receita de dez/2017 vs out/2017: {m_idx.loc['2017-12', 'receita'] / m_idx.loc['2017-10', 'receita'] - 1:+.1%}")

In [ ]:
# Dia da semana
dias = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sáb", "Dom"]
dow = (
    pedidos_entregues.assign(dia_semana=pedidos_entregues["order_purchase_timestamp"].dt.dayofweek)
    .groupby("dia_semana").agg(pedidos=("order_id", "nunique"), receita=("receita_produtos", "sum"))
)
dow["participacao_pct"] = dow["receita"] / receita_total * 100
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(dias, dow["participacao_pct"], color="#7C3AED")
ax.set(title="Participação na receita por dia da semana", ylabel="% da receita")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()
print(f"Segunda a sexta: {dow['participacao_pct'].iloc[:5].sum():.1f}% da receita | fim de semana: {dow['participacao_pct'].iloc[5:].sum():.1f}%")

## 12. Previsão de receita (set–dez/2018)

Antes de projetar, **testamos três métodos de previsão** com um backtest: treinamos com jan/2017–fev/2018 e comparamos com o que aconteceu de mar a ago/2018. O método com menor erro (MAPE) vira o cenário-base. Depois aplicamos a sazonalidade de outubro/novembro/dezembro observada em 2017 (que é só um ano de histórico, portanto é uma hipótese, não uma certeza).

Cenários:
- **Base:** receita mensal estabiliza no patamar médio dos últimos 6 meses.
- **Conservador:** continua a leve tendência de queda observada em 2018.
- **Otimista:** retoma o ritmo de crescimento linear de jan/2017 a ago/2018 (o backtest mostra que esse método tende a superestimar).

In [ ]:
# Backtest: treino jan/2017–fev/2018 (14 meses), teste mar–ago/2018 (6 meses)
y = mensal_comparavel["receita"].values
t = np.arange(len(y))
n_treino, n_teste = 14, 6

b, a = np.polyfit(t[:n_treino], y[:n_treino], 1)
prev_linear = a + b * t[n_treino:]
prev_patamar = np.full(n_teste, y[n_treino - 6:n_treino].mean())
b2, a2 = np.polyfit(t[n_treino - 6:n_treino], y[n_treino - 6:n_treino], 1)
prev_tend_recente = a2 + b2 * (t[n_treino:] - (n_treino - 6))

def mape(prev):
    return np.mean(np.abs(prev - y[n_treino:]) / y[n_treino:]) * 100

backtest = pd.DataFrame({
    "Método": ["Patamar dos últimos 6 meses", "Tendência linear (todo o período)", "Tendência linear (últimos 6 meses)"],
    "Erro médio percentual (MAPE, %)": [mape(prev_patamar), mape(prev_linear), mape(prev_tend_recente)],
}).sort_values("Erro médio percentual (MAPE, %)")
display(backtest.round(1))

In [ ]:
# Cenários set–dez/2018
meses_prev = pd.period_range("2018-09", "2018-12", freq="M")
h = np.arange(len(y), len(y) + len(meses_prev))

b_full, a_full = np.polyfit(t, y, 1)
cen_otimista = a_full + b_full * h
y18 = y[-8:]
b18, a18_ = np.polyfit(np.arange(8), y18, 1)
cen_conservador = a18_ + b18 * np.arange(8, 8 + len(meses_prev))
cen_base = np.full(len(meses_prev), y[-6:].mean())

# Sazonalidade observada em 2017 em relação a outubro (nov = Black Friday, dez = Natal)
m_rec = mensal.set_index("ano_mes")["receita"]
fator = np.array([1.0, 1.0, m_rec["2017-11"] / m_rec["2017-10"], m_rec["2017-12"] / m_rec["2017-10"]])
print("Fatores sazonais (set, out, nov, dez):", fator.round(2))

previsao = pd.DataFrame({
    "Conservador": cen_conservador * fator,
    "Base": cen_base * fator,
    "Otimista": cen_otimista * fator,
}, index=meses_prev.astype(str))
display(previsao.round(0))
print("Total set–dez/2018 (R$):")
display(previsao.sum().round(0).to_frame("Receita total prevista"))

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(mensal_comparavel["mes"], mensal_comparavel["receita"] / 1e3, marker="o", color="#0F766E", label="Realizado")
for nome, cor in [("Conservador", "#DC2626"), ("Base", "#2563EB"), ("Otimista", "#16A34A")]:
    ax.plot(previsao.index, previsao[nome] / 1e3, marker="o", linestyle="--", color=cor, label=nome)
ax.set(title="Receita mensal: realizado e cenários de previsão", ylabel="Receita (R$ mil)")
ax.tick_params(axis="x", rotation=75)
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 13. Conclusões e recomendações para investidores

### O que os dados mostram
1. **Crescimento forte, mas concentrado em 2017.** Entre jan–ago/2017 e jan–ago/2018 a receita de produtos cresceu **141%** (de R\$ 3,0 mi para R\$ 7,2 mi) e os pedidos **140%**. O ticket médio ficou praticamente igual (+0,5%). Ou seja, o crescimento veio de **volume**, não de pedidos maiores.
2. **Em 2018 o crescimento mensal estagnou.** Desde jan/2018 a receita mensal oscila entre R\$ 0,83 mi e R\$ 0,98 mi, e jun–ago ficaram abaixo de mai. O patamar médio de mar–ago/2018 é de R\$ 0,91 mi por mês.
3. **Um dia concentra a sazonalidade.** Na Black Friday de 2017 foram **1.147 pedidos em um único dia (6,1x a média diária anterior)**. Novembro teve receita **52% maior** que outubro e dezembro **12% maior**.
4. **Receita concentrada em poucas categorias, regiões e sellers.** 18 de 74 categorias (24%) geram 80% da receita. SP, RJ e MG somam **63,4%** (SP sozinho 38,3%). Os 10% maiores sellers respondem por **67%** da receita.
5. **A logística é um ponto de atrito.** Os sellers de SP geram **64,4%** da receita, mas só 38,3% dos clientes estão em SP. Apenas 31,3% da receita é de seller e cliente no mesmo estado. O frete equivale a **18,3%** do preço quando o envio é entre estados diferentes, contra **13,0%** no mesmo estado. O frete mediano é 18% do valor do pedido.
6. **Quase não há recompra.** Só **3,0%** dos clientes compraram mais de uma vez (5,5% da receita), e **90%** dos pedidos têm apenas 1 item.
7. **A média engana:** o ticket médio é R\$ 137,04, mas a mediana é R\$ 86,57 (poucos pedidos muito caros puxam a média, e o coeficiente de variação de 153%).

### Previsão (set–dez/2018)
O backtest escolheu o método mais simples (patamar dos últimos 6 meses, erro médio de 13,3%) em vez das tendências lineares (23–25% de erro). Com a sazonalidade de 2017, a receita de set–dez/2018 seria de aproximadamente **R\$ 4,2 mi no cenário-base** (R\$ 4,0 mi no conservador e R\$ 5,4 mi no otimista), com novembro como o mês mais forte. É uma estimativa com apenas um ano de sazonalidade observada, e não uma garantia.

### Recomendações
| # | Ação | Evidência | Indicador para acompanhar |
|---|---|---|---|
| 1 | **Programa de retenção e recompra** (CRM, cupom pós-compra, assinatura) | Só 3% recompram | Taxa de recompra em 90 dias |
| 2 | **Reduzir o custo logístico entre estados** (centros de distribuição ou parceiros fora de SP, frete grátis acima de um valor mínimo) | Frete pesa 18% do preço entre estados e 64% dos sellers em SP | Frete ÷ preço e conversão no Nordeste e Norte |
| 3 | **Calendário promocional** além da Black Friday (datas com preparação de estoque e capacidade) | Um dia = 6x a demanda normal e 2018 estagnado | Receita mensal vs patamar de R\$ 0,91 mi |
| 4 | **Aumentar itens por pedido** (cross-sell, kits, frete grátis por faixa) | 90% dos pedidos têm 1 item | Itens por pedido e ticket médio |
| 5 | **Reduzir a dependência de poucos sellers** (programa de desenvolvimento dos sellers intermediários) | Top 10% = 67% da receita | Receita do top 10% e nº de sellers ativos |
| 6 | **Campanhas de fim de semana** | Sáb/dom = 22,8% da receita vs 28,6% se fosse uniforme | Receita de sáb/dom |